# Step 4. 할인 이력 수집

**목표**: 50개 게임의 Steam 할인 이력 수집 (최근 2년)  
**입력**: `data/game_metadata.csv`  
**출력**: `data/discount_history.csv`  
**API**: IsThereAnyDeal (ITAD) v2 — Bearer 토큰 인증

### 처리 흐름
1. Steam appid → ITAD 게임 ID 조회
2. ITAD 게임 ID → Steam 가격 히스토리 조회
3. 가격 스냅샷 → 할인 이벤트 (시작일/종료일/할인율) 변환

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
from tqdm.auto import tqdm
from dotenv import load_dotenv
import os

load_dotenv()
ITAD_API_KEY = os.getenv("ITAD_API_KEY")
print("ITAD_API_KEY:", "OK" if ITAD_API_KEY else "MISSING (.env 확인 필요)")

df = pd.read_csv("../data/game_metadata.csv")
print(f"게임 수: {len(df)}개")

In [ ]:
SINCE_DT = datetime.now(timezone.utc) - timedelta(days=730)
SINCE_ISO = SINCE_DT.strftime("%Y-%m-%dT%H:%M:%S+00:00")  # ITAD API 필수 포맷

print(f"수집 기간: {SINCE_DT.strftime('%Y-%m-%d')} ~ 현재")
print(f"since 파라미터: {SINCE_ISO}")

## API 연결 진단

본격 수집 전에 Terraria 1개로 API 연결 테스트.

In [ ]:
resp = requests.get(
    "https://api.isthereanydeal.com/games/lookup/v1",
    params={"key": ITAD_API_KEY, "appid": 105600},
    timeout=10
)
print(f"status: {resp.status_code}")
print(f"response: {resp.text[:300]}")

## 헬퍼 함수

In [ ]:
def lookup_itad_id(appid):
    """Steam appid로 ITAD 게임 ID 조회"""
    try:
        resp = requests.get(
            "https://api.isthereanydeal.com/games/lookup/v1",
            params={"key": ITAD_API_KEY, "appid": appid},
            timeout=10
        )
        if resp.status_code == 404:
            return None
        resp.raise_for_status()
        return resp.json().get("game", {}).get("id")
    except Exception:
        return None


def get_price_history(game_id, since_iso):
    """ITAD에서 Steam 가격 히스토리 조회 (전체 받아서 Steam만 필터)"""
    try:
        resp = requests.get(
            "https://api.isthereanydeal.com/games/history/v2",
            params={"key": ITAD_API_KEY, "id": game_id, "since": since_iso},
            timeout=15
        )
        if resp.status_code in (400, 404):
            return []
        resp.raise_for_status()
        data = resp.json() or []
        # Steam 스토어 항목만 필터
        return [x for x in data if x.get("shop", {}).get("name") == "Steam"]
    except Exception:
        return []


def snapshots_to_events(snapshots, appid, name, genre):
    """가격 스냅샷 → 할인 이벤트 변환 (cut > 0 구간을 하나의 이벤트로 묶음)"""
    if not snapshots:
        return []

    snapshots = sorted(snapshots, key=lambda x: x.get("timestamp", ""))
    events = []
    in_discount = False
    discount_start = None
    discount_pct = 0

    for snap in snapshots:
        cut = snap.get("deal", {}).get("cut", 0)
        ts_str = snap.get("timestamp", "")
        try:
            dt = datetime.fromisoformat(ts_str)
        except Exception:
            continue

        if cut > 0 and not in_discount:
            in_discount = True
            discount_start = dt
            discount_pct = cut
        elif cut == 0 and in_discount:
            in_discount = False
            discount_end = dt
            events.append({
                "appid": appid, "name": name, "genre_category": genre,
                "discount_start": discount_start.strftime("%Y-%m-%d"),
                "discount_end": discount_end.strftime("%Y-%m-%d"),
                "discount_pct": discount_pct,
                "duration_days": (discount_end - discount_start).days,
            })
            discount_start = None
            discount_pct = 0

    if in_discount and discount_start:
        today = datetime.now(timezone.utc)
        events.append({
            "appid": appid, "name": name, "genre_category": genre,
            "discount_start": discount_start.strftime("%Y-%m-%d"),
            "discount_end": today.strftime("%Y-%m-%d"),
            "discount_pct": discount_pct,
            "duration_days": (today - discount_start).days,
        })

    return events

print("함수 정의 완료")

## 할인 이력 수집

게임당 API 2번 호출 (ID 조회 + 이력 조회), 약 2분 소요.

In [ ]:
all_events = []
failed = []
no_data = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="할인 이력 수집"):
    appid = int(row["appid"])
    name  = row["name"]
    genre = row["genre_category"]

    itad_id = lookup_itad_id(appid)
    time.sleep(1.0)

    if not itad_id:
        failed.append({"appid": appid, "name": name})
        continue

    snapshots = get_price_history(itad_id, SINCE_ISO)
    time.sleep(1.0)

    if not snapshots:
        no_data.append({"appid": appid, "name": name})
        continue

    all_events.extend(snapshots_to_events(snapshots, appid, name, genre))

print(f"\n수집 완료")
print(f"  할인 이벤트: {len(all_events)}개")
print(f"  ITAD 조회 실패: {len(failed)}개")
print(f"  Steam 할인 이력 없음: {len(no_data)}개")

if failed:
    print("\nITAD 조회 실패:")
    for g in failed: print(f"  [{g['appid']}] {g['name']}")
if no_data:
    print("\n할인 이력 없음:")
    for g in no_data: print(f"  [{g['appid']}] {g['name']}")

## 결과 확인

In [ ]:
result_df = pd.DataFrame(all_events)

print("장르별 할인 이벤트 수:")
print(result_df.groupby("genre_category")["appid"].count().rename("이벤트 수"))
print(f"\n총 {len(result_df)}개 할인 이벤트")
print(f"할인율 범위: {result_df['discount_pct'].min()}% ~ {result_df['discount_pct'].max()}%")
print(f"평균 할인율: {result_df['discount_pct'].mean():.1f}%")
print(f"평균 할인 기간: {result_df['duration_days'].mean():.1f}일")
result_df.head(20)

In [ ]:
events_per_game = (
    result_df.groupby(["appid", "name", "genre_category"])["discount_pct"]
    .count().reset_index()
    .rename(columns={"discount_pct": "event_count"})
    .sort_values("event_count")
)
print(f"이벤트 2개 미만 게임: {(events_per_game['event_count'] < 2).sum()}개")
events_per_game

## CSV 저장

In [ ]:
output_path = "../data/discount_history.csv"
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")
print(f"Shape: {result_df.shape}")